In [16]:
#!pip install pandas gradio openai==0.28.0 sentence-transformers pinecone google-generativeai groq


In [26]:
import re
import pandas as pd
import gradio as gr
import openai
from sentence_transformers import SentenceTransformer
from pinecone import pinecone

INDEX_NAME = "matching"

pc = Pinecone(api_key='pcsk_4GsudP_3tuAHhmxssWrKmXVhPLzBVa68pYhMQmskHwnKBCtmEPHQ7UAPC3UF5LPDAXm921')
index = pc.Index(
    host='https://matching-uvifknu.svc.aped-4627-b74a.pinecone.io'
)

from google.generativeai import GenerativeModel, configure
from groq import Groq  # New import for Groq

# Extend the LLM models dictionary to include Deepseek.
llm_models = {
    "GPT-3.5": "gpt-3.5-turbo-0125",
    "GPT-4": "gpt-4-turbo",
    "Gemini Pro": "gemini-1.5-flash",
    "Llama 3.3": "llama-3.3-70b-versatile",
    "Deepseek": "deepseek-r1-distill-llama-70b"
}

def remove_think_block(text: str) -> str:
    """
    Removes any content (including the tags) enclosed between <think> and </think>.
    """
    text=re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return (text.strip())

def RAG(prompt_text):
    model = SentenceTransformer("all-mpnet-base-v2")
    embeddings = model.encode(prompt_text)
    context = gradio_interface(embeddings)
    return context

df = pd.read_csv("Data.csv", encoding='cp1252')

def gradio_interface(context):
    top_k = 2
    vec = context.tolist()
    embeddings = index.query(
        vector=vec,
        top_k=top_k,
        include_values=True
    )
    all_values = [match['id'] for match in embeddings['matches']]
    ret = "\nHere is the first set of criteria, patient profile and result on eligibility:"
    for j in range(top_k):
        print("Id is ", all_values[j])
        i = int(all_values[j])
        r = ""
        r += "\n Inclusion Criteria for the trial is \n" + df['inclusion_criteria'][i]
        r += "\n Exclusion Criteria for the trial is \n" + df['exclusion_criteria'][i]
        r += "\n Patient Profile is \n" + df['patient_profiles'][i]
        r += "\n The eligibility of the patient for the clinical trial is: \n" + df['target'][i]
        if j != top_k-1:
            r += " \n Here is another patient profile: "
        ret += r
    return ret

def process_gemini(curr_string, api_key):
    configure(api_key=api_key)
    model = GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(curr_string)
    response_text = response.text if hasattr(response, 'text') else response[0]['text']
    verdict = response_text.split('\n')[0]
    reasoning = "\n".join(response_text.split('\n')[1:])
    return verdict, reasoning

def process_llama(curr_string, api_key, temp_in):
    # Instantiate Groq client using Llama 3.3 model.
    client = Groq(api_key=api_key)
    completion = client.chat.completions.create(
        model=llm_models["Llama 3.3"],
        messages=[
            {"role": "system", "content": "You are a helpful assistant and acting as an oncologist"},
            {"role": "user", "content": curr_string}
        ],
        temperature=temp_in,
        max_completion_tokens=15000,
        top_p=1,
        stream=False,
        stop=None
    )
    response_text = completion.choices[0].message.content  # use attribute access
    verdict = response_text.split("\n")[0]
    llm_reasoning = "\n".join(response_text.split("\n")[1:])
    return verdict, llm_reasoning

def process_deepseek(curr_string, api_key, temp_in):
    # Instantiate Groq client using the Deepseek model.
    client = Groq(api_key=api_key)
    completion = client.chat.completions.create(
        model=llm_models["Deepseek"],
        messages=[
            {"role": "system", "content": "You are a helpful assistant and acting as an oncologist"},
            {"role": "user", "content": curr_string}
        ],
        temperature=temp_in,
        max_completion_tokens=15000,
        top_p=1,
        stream=False,
        stop=None
    )
    response_text = completion.choices[0].message.content
    # Remove any chain-of-thought blocks enclosed in <think>...</think>
    response_text = remove_think_block(response_text)
    verdict = response_text.split("\n")[0]
    llm_reasoning = "\n".join(response_text.split("\n")[1:])
    return verdict, llm_reasoning

def process_text(input1, input2, input3, api_key, model_in, temp_in, rag_check):
    print("RAG :", rag_check)
    curr_string = (
        f"You are an oncologist. If for a clinical trial, the Inclusion Criteria is :\n{input2}\n"
        f"and the Exclusion Criteria is : {input3}\n"
        f"and the Patient Profile is :\n{input1}\n"
        ".\n Then given this information determine if the patient is eligible for the trial. "
        "Give a binary answer Eligible/Not Eligible as the response. In the response please do not add any extra text, "
        "just the response Eligible or Not Eligible. After giving the verdict in one line, from the next line give the reasons for giving the verdict as eligible or not eligible"
    )

    if rag_check:
        query = RAG(curr_string)
        curr_string += "\nTo determine the eligibility, you may consider as an example, some similar patients with inclusion criteria, exclusion criteria, and a verdict on its eligibility below to help you better make your decision about the eligibility:\n"
        curr_string += query

    if model_in == "Gemini Pro":
        verdict, llm_reasoning = process_gemini(curr_string, api_key)
    elif model_in == "Llama 3.3":
        verdict, llm_reasoning = process_llama(curr_string, api_key, temp_in)
    elif model_in == "Deepseek":
        verdict, llm_reasoning = process_deepseek(curr_string, api_key, temp_in)
    else:
        openai.api_key = api_key
        response = openai.ChatCompletion.create(
            model=llm_models[model_in],
            messages=[
                {"role": "system", "content": "You are a helpful assistant and acting as an oncologist"},
                {"role": "user", "content": curr_string}
            ],
            temperature=temp_in
        )
        verdict = response["choices"][0]["message"]["content"].split("\n")[0]
        llm_reasoning = "\n".join(response["choices"][0]["message"]["content"].split("\n")[1:])
    return verdict, llm_reasoning

# Gradio input and output definitions
input_box1 = gr.Textbox(label="Patient Profile")
input_box2 = gr.Textbox(label="Inclusion Criteria")
input_box3 = gr.Textbox(label="Exclusion Criteria")
input_box4 = gr.Textbox(label="API Key")  # Single input box for API key

output_box1 = gr.Textbox(label="Verdict")
output_box2 = gr.Textbox(label="LLM Reason")

with gr.Blocks() as demo:
    big_block = gr.HTML("""
    <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/8/84/Syneos_Health_logo.svg/1200px-Syneos_Health_logo.svg.png" alt="logo" width="150" height="150">
    """)
    gr.Interface(
        fn=process_text,
        inputs=[
            input_box1, input_box2, input_box3, input_box4,
            gr.Dropdown(list(llm_models.keys()), label="LLM Model", info="Select a LLM model"),
            gr.Slider(0, 1, value=0.9, label="Temperature", info="Select a Temperature"),
            gr.Checkbox(label="Apply RAG", info="Check the following to use RAG", value=True)
        ],
        outputs=[output_box1, output_box2],
        title="Predict Patient Eligibility",
        description="Enter Patient Profile and Inclusion and Exclusion criteria to check Eligibility of patient",
        allow_flagging="never"
    )
demo.launch(debug=True)


/usr/local/lib/python3.11/dist-packages/gradio/interface.py:403: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated.Use `flagging_mode` instead.
  warnings.warn(


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://926974c62e736f0e0c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


RAG : True
Id is  0
Id is  1
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://926974c62e736f0e0c.gradio.live
